<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# 8. Reflexion Agent (Self-Critique Loop)
**Industry:** Consulting Services

Build an agent that answers a client question/brief, critiques its own answer, and revises it — repeating until a quality threshold or max iterations is reached.

In [1]:
!pip install langgraph langchain langchain-openai pydantic


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
from typing import TypedDict
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
dotenv.load_dotenv(r"D:/Internship/Teach-ai/Backend/.env")
from langchain_openai import AzureChatOpenAI
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

llm = AzureChatOpenAI(azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"), api_key=os.environ.get("AZURE_OPENAI_API_KEY"), azure_deployment="gpt-4o", api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"))

class Critique(BaseModel):
    feedback: str = Field(description="Constructive criticism of the answer")
    is_good_enough: bool = Field(description="True if the answer meets professional consulting standards, False otherwise")

critique_llm = llm.with_structured_output(Critique)

class ReflexionState(TypedDict):
    question: str
    draft: str
    feedback: str
    iterations: int

def draft_node(state: ReflexionState):
    if state.get("feedback"):
        prompt = f"Revise this draft based on the feedback.\n\nDraft: {state['draft']}\nFeedback: {state['feedback']}"
    else:
        prompt = f"Draft an initial consulting answer for: {state['question']}"
    response = llm.invoke(prompt)
    return {"draft": response.content, "iterations": state.get("iterations", 0) + 1}

def critique_node(state: ReflexionState):
    prompt = f"Critique this consulting strategy.\nQuestion: {state['question']}\nDraft: {state['draft']}"
    eval = critique_llm.invoke(prompt)
    return {"feedback": eval.feedback, "is_good_enough": eval.is_good_enough}

def route(state: ReflexionState):
    if state.get("is_good_enough") or state["iterations"] >= 3:
        return END
    return "draft_node"

workflow = StateGraph(ReflexionState)
workflow.add_node("draft_node", draft_node)
workflow.add_node("critique_node", critique_node)

workflow.add_edge(START, "draft_node")
workflow.add_edge("draft_node", "critique_node")
workflow.add_conditional_edges("critique_node", route, ["draft_node", END])

app = workflow.compile()

inputs = {"question": "Outline a market-entry strategy for a new EV brand in India.", "iterations": 0}
for event in app.stream(inputs):
    for k, v in event.items():
        if k == "draft_node":
            print(f"\n[Draft Iteration {v['iterations']}]:\n", v['draft'][:200], "...")
        elif k == "critique_node":
            print(f"\n[Critique]:\n", v['feedback'])
            print("Pass?", v['is_good_enough'])


[Draft Iteration 1]:
 **Draft Consulting Answer: Market Entry Strategy for a New EV Brand in India**

Entering the Indian electric vehicle (EV) market presents significant opportunities, given the government's push for sus ...



[Critique]:
 This strategy is comprehensive, addressing key market dynamics, infrastructure challenges, customer perception, and strategic partnerships. However, it could further elaborate on the competitive edge through distinct branding and innovation approaches, as well as a deeper segmentation analysis for targeting.


KeyError: 'is_good_enough'